In [1]:
#format the book
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 交互演示（Interactions）

这是书中交互演示的集合。若你阅读纸质版，或通过 Github 或 nbviewer 在线阅读，将无法运行这些交互。

因此我创建了本 notebook。若你的电脑未安装 IPython，可按以下方式运行交互：

1. 在浏览器中打开 try.juptyer.org，它会为你启动一个临时 notebook 服务器。

2. 点击 **New** 按钮，选择 `Python 3`。这会在浏览器中创建一个运行 Python 3 的新 notebook。

3. 从本 notebook 复制某个单元格的全部内容，粘贴到浏览器 notebook 的 code 单元格中。

4. 按 CTRL+ENTER 执行该单元格。

5. 尽情尝试！改代码、玩耍、实验、折腾。

你的服务器与 notebook 不会永久保存。关闭会话后数据即丢失。是的，按保存会显示正在保存，目录里也能看到文件，但那只是在 Docker 容器里，关闭窗口后会被删除。若想保留修改，请复制粘贴到外部文件。

当然，若已安装 IPython，可下载本 notebook 在本地运行。在下载文件的目录下于命令行输入

    ipython notebook
    
点击本文件名即可打开。



# 试验 FPF'

卡尔曼滤波（Kalman filter）在预测（prediction）步用方程 $P^- = FPF^\mathsf{T}$ 计算协方差矩阵（covariance matrix）的先验（prior），其中 $P$ 是协方差矩阵，$F$ 是系统转移函数。对牛顿系统 $x = \dot{x}\Delta t + x_0$，$F$ 可能形如

$$F = \begin{bmatrix}1 & \Delta t\\0 & 1\end{bmatrix}$$

$FPF^\mathsf{T}$ 通过位置（$x$）与速度（$\dot{x}$）之间的相关性改变 $P$。本交互图可让你看到不同 $F$ 设计对该值的影响。例如：

* 若 $x$ 与 $\dot{x}$ 不相关？（将 F01 设为 0）

* 若 $x = 2\dot{x}\Delta t + x_0$？（将 F01 设为 2）

* 若 $x = \dot{x}\Delta t + 2*x_0$？（将 F00 设为 2）

* 若 $x = \dot{x}\Delta t$？（将 F00 设为 0）



In [2]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import numpy as np
import numpy.linalg as linalg
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(x, P, edgecolor='k'):
    U,s,v = linalg.svd(P)
    angle = math.atan2(U[1,0],U[0,0])
    width  = math.sqrt(s[0]) * 2
    height = math.sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor=edgecolor, facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    ax.set_aspect('equal')
    
    
def plot_FPFT(F00, F01, F10, F11, covar):
    
    dt = 1.
    x = np.array((0, 0.))
    P = np.array(((1, covar), (covar, 2)))
    F = np.array(((F00, F01), (F10, F11)))

    plot_covariance_ellipse(x, P)
    plot_covariance_ellipse(x, np.dot(F, P).dot(F.T), edgecolor='r')
    #plt.axis('equal')
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.title(str(F))
    plt.xlabel('position')
    plt.ylabel('velocity')
    plt.show()
                 
interact(plot_FPFT, 
         F00=widgets.IntSlider(value=1, min=0, max=2.), 
         F01=widgets.FloatSlider(value=1, min=0., max=2., description='F01(dt)'),
         F10=widgets.FloatSlider(value=0, min=0., max=2.),
         F11=widgets.FloatSlider(value=1, min=0., max=2.),
         covar=widgets.FloatSlider(value=0, min=0, max=1.));

interactive(children=(IntSlider(value=1, description='F00', max=2), FloatSlider(value=1.0, description='F01(dt…

# 协方差椭圆（Covariance Ellipse）

观察如下形式协方差矩阵的方差与协方差变化带来的影响：

$$\begin{bmatrix}\texttt{var}_x & \texttt{cov}_xy \\ \texttt{cov}_xy & \texttt{var}_y\end{bmatrix}$$



In [3]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
from ipywidgets import FloatSlider
from math import cos, sin, pi, atan2, sqrt
import  numpy.linalg as linalg
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_covariance_ellipse(P):
    U,s,v = linalg.svd(P)
    angle = atan2(U[1,0],U[0,0])
    width  = sqrt(s[0]) * 2
    height = sqrt(s[1]) * 2

    ax = plt.gca()
    e = Ellipse(xy=(0, 0), width=width, height=height, angle=angle,
                edgecolor='k', facecolor='none',
                lw=2, ls='solid')
    ax.add_patch(e)
    h, w = height/4, width/4
    plt.plot([0, h*cos(angle+pi/2)], [0, h*sin(angle+pi/2)])
    plt.plot([0, w*cos(angle)],      [0, w*sin(angle)])

def plot_covariance(var_x, var_y, cov_xy):
    P = [[var_x, cov_xy], [cov_xy, var_y]]
    plot_covariance_ellipse(P)
    plt.xlim(-6, 6)
    plt.gca().set_aspect('equal')
    plt.ylim(-6, 6)
    plt.show()

interact (plot_covariance,           
          var_x=FloatSlider(value=5., min=0, max=20.), 
          var_y=FloatSlider(value=5., min=0., max=20.), 
          cov_xy=FloatSlider(value=1.5, min=0.0, max=50, step=.2));


interactive(children=(FloatSlider(value=5.0, description='var_x', max=20.0), FloatSlider(value=5.0, descriptio…

# g-h 滤波（g-h Filter）

试验 g-h 滤波（g-h filter）各参数的不同取值。



In [4]:
%matplotlib inline
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import numpy.random as random

def gen_data(x0, dx, count, noise_factor):
    return [x0 + dx*i + random.randn()*noise_factor for i in range (count)]

def g_h_filter(data, x0, dx, g, h, dt=1., pred=None):    
    x = x0
    results = []
    for z in data:
        #prediction step
        x_est = x + (dx*dt)
        dx = dx        
        if pred is not None:
            pred.append(x_est)
        
        # update step
        residual = z - x_est
        dx = dx    + h * (residual) / dt
        x  = x_est + g * residual     
        results.append(x)  
    return np.array(results)

zs = gen_data(x0=5, dx=5, count=100, noise_factor=50)

def interactive_gh(x, dx, g, h):
    data = g_h_filter(data=zs, x0=x, dx=dx, dt=1.,g=g, h=h)
    plt.plot(zs, color='r')
    plt.plot(data, color='k')
    plt.show()

interact (interactive_gh,           
          x=widgets.FloatSlider(value=0., min=-50, max=50.), 
          dx=widgets.FloatSlider(value=5., min=-50., max=50.), 
          g=widgets.FloatSlider(value=0.1, min=0.01, max=2, step=.02), 
          h=widgets.FloatSlider(value=0.02, min=0.0, max=0.5, step=0.01));

interactive(children=(FloatSlider(value=0.0, description='x', max=50.0, min=-50.0), FloatSlider(value=5.0, des…